# Mortgage Approval Workflow with Oracle AI Agent Memory

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/oracle-devrel/oracle-ai-developer-hub/blob/main/notebooks/agent_memory/03_mortgage_workflow_langgraph.ipynb) [![Documentation](https://img.shields.io/badge/Documentation-Oracle%20AI%20Agent%20Memory-red?style=flat-square)](https://www.oracle.com/database/ai-agent-memory/)


**Framework:** LangGraph (prebuilt `create_agent` nodes inside a `StateGraph`) · **Memory:** Oracle AI Agent Memory on Oracle AI Database

This notebook builds a **mortgage approval workflow** — a predetermined pipeline that runs every application through the same sequence of checks: identity verification, credit assessment, income verification, debt-to-income calculation, and final decision. The workflow is deterministic; the LLM reasoning only happens inside individual nodes.

---

## Application Modes in Agent Applications

AI agent applications generally fall into three operational modes:

| Mode | Shape | Typical use |
|---|---|---|
| **Assistant** | Turn-by-turn conversational, tool-using | Operations support, customer service, internal copilots |
| **Deep Research** | Multi-step autonomous investigation | Literature review, scoping, market research |
| **Workflow** | Predetermined sequence of steps with conditional branches | Approval pipelines, compliance gates, structured triage |

> **📌 This notebook focuses on the Workflow mode.**
>
> In Workflow mode, the *shape* of the work is fixed up-front — you know the stages, their order, and the conditions that route between them. The LLM does the reasoning inside each stage, but the orchestration is deterministic and auditable. That determinism is exactly what regulated domains like mortgage lending require.

## What You'll Learn

- How to model a deterministic workflow as a **`StateGraph`** with typed state
- How to use LangChain's **prebuilt `create_agent`** as an intelligent node inside the graph
- How to drive conditional routing with **`add_conditional_edges`**
- How to use Oracle AI Agent Memory to persist applicant data and audit trails across the workflow's stages and across runs

> **💡 Key insight:** Memory is what lets a workflow survive a crash or a restart. When stage 3 fails mid-run, the state up to stage 3 is already in memory — the workflow can resume rather than restart. That is the practical difference between a memory-aware workflow and a script.

## Prerequisites

- **Oracle AI Database** running locally (Docker container) or reachable over the network
- **`OPENAI_API_KEY`** for the agent LLM and embeddings
- The `oracleagentmemory` wheel installed in the active kernel's environment

## 1. Install dependencies

In [1]:
%pip install -q "oracleagentmemory[litellm]" langgraph langchain langchain-openai nest_asyncio

Note: you may need to restart the kernel to use updated packages.


## 2. Configure API keys and Oracle connection

In [ ]:
import os

os.environ.setdefault("OPENAI_API_KEY", "sk-")
os.environ.setdefault("DB_USER", "VECTOR")
os.environ.setdefault("DB_PASSWORD", "VectorPwd_2025")
os.environ.setdefault("DB_CONNECT_STRING", "localhost:1521/FREEPDB1")

import nest_asyncio
nest_asyncio.apply()
print("Environment configured.")

Environment configured.


## 3. Connect to Oracle AI Database and initialise the memory client

The workflow uses memory for three things:

1. **Applicant records** — identity, financial data, third-party verifications — stored as durable memories keyed by applicant ID.
2. **Stage outcomes** — each node writes its decision and rationale, producing an auditable trail.
3. **Policy facts** — underwriting thresholds, approved document types — stored as `fact` records the agent nodes can recall.

> **💡 Key insight:** In a regulated workflow, memory is also your audit trail. Every stage's input, decision, and rationale should be persisted in a queryable substrate. Oracle AI Database gives you that substrate and the compliance controls already wrapped around it.

In [3]:
import oracledb
from oracleagentmemory.core import OracleAgentMemory
from oracleagentmemory.core.llms import Llm

connection = oracledb.connect(
    user=os.environ["DB_USER"],
    password=os.environ["DB_PASSWORD"],
    dsn=os.environ["DB_CONNECT_STRING"],
)
print("Connected to Oracle AI Database:", connection.version)

extraction_llm = Llm("gpt-4o-mini", temperature=0.1)

memory_client = OracleAgentMemory(
    connection=connection,
    embedder="text-embedding-3-small",
    llm=extraction_llm,
    extract_memories=False,          # workflow writes memories explicitly
    schema_policy="create_if_necessary",
    table_name_prefix="mortgage_",
)
print("Memory client ready.")

Connected to Oracle AI Database: 23.26.0.0.0
Memory client ready.


## 4. Register the underwriter (user) and the workflow agent

In a mortgage pipeline, the `user_id` typically represents the loan officer or underwriter who owns the file. The `agent_id` represents the workflow itself, so the audit trail is attributed to the pipeline version and owned by the right human.

In [4]:
UNDERWRITER_ID = "underwriter-alex"
AGENT_ID = "mortgage-workflow-v1"

for create_fn, eid, info in [
    (memory_client.add_user, UNDERWRITER_ID, "Senior underwriter — residential mortgages."),
    (memory_client.add_agent, AGENT_ID, "Deterministic mortgage approval workflow (v1)."),
]:
    try:
        create_fn(eid, info)
        print(f"Registered: {eid}")
    except ValueError:
        print(f"Already exists: {eid}")

Already exists: underwriter-alex
Already exists: mortgage-workflow-v1


## 5. Seed underwriting policy as durable facts

Underwriting thresholds (minimum credit score, maximum DTI) don't live in code — they're policy. We store them as `fact` records so the agent nodes can recall them via search rather than hard-coding them, and so policy changes don't require code changes.

> **📌 Production pattern.** `fact` records are a first-class OAMP record type. They're indexed and searchable the same way memories are, but conceptually represent static reference data rather than learned information. Treat them as the memory layer's equivalent of configuration.

In [5]:
store = memory_client._store   # fact/guideline convenience methods live on the store

POLICY_FACTS = [
    "Minimum acceptable credit score for conventional loans is 620.",
    "Maximum acceptable debt-to-income ratio is 43%.",
    "Accepted income documentation: W-2 forms, 1040 returns, pay stubs from the last 30 days.",
    "Identity verification requires a government-issued photo ID and a utility bill or bank statement.",
    "Loans above $1.5M require manual senior-underwriter review regardless of automated decision.",
]

fact_ids = store.add(
    contents=POLICY_FACTS,
    record_type="fact",
    user_ids=UNDERWRITER_ID,
    agent_ids=AGENT_ID,
)
print(f"Seeded {len(fact_ids)} policy facts.")

Seeded 5 policy facts.


## 6. Define the workflow state

LangGraph uses a typed state dict (`TypedDict`) that each node mutates by returning partial updates. For a mortgage workflow, the state carries everything the pipeline accumulates — applicant data, verification outcomes, and the final decision.

> **💡 Key insight:** Typed state is what makes a LangGraph workflow debuggable. You can introspect any run by dumping the state; any audit question ("what was the DTI at decision time?") is answerable from state alone.

In [6]:
from typing_extensions import TypedDict
from typing import Optional

class MortgageState(TypedDict, total=False):
    # Input
    applicant_id: str
    applicant_name: str
    loan_amount: float
    property_value: float
    # Stage outputs
    identity_verified: bool
    identity_notes: str
    credit_score: int
    credit_notes: str
    monthly_income: float
    monthly_debt: float
    income_notes: str
    dti_ratio: float
    # Decision
    decision: str        # "approve" | "deny" | "manual_review"
    decision_reason: str

print("MortgageState defined with", len(MortgageState.__annotations__), "fields.")

MortgageState defined with 14 fields.


## 7. Helper — write a durable audit memory for each stage

Every node calls this helper so every stage's decision is persisted as a memory with consistent metadata. That gives us a complete audit trail per applicant, searchable semantically (for pattern analysis) and filterable by applicant ID (for file reviews).

In [7]:
def log_stage(applicant_id: str, stage: str, outcome: str, rationale: str) -> str:
    """Write a structured audit memory for one workflow stage."""
    content = f"[{stage}] applicant={applicant_id} outcome={outcome}. {rationale}"
    return memory_client.add_memory(
        content,
        user_id=UNDERWRITER_ID,
        agent_id=AGENT_ID,
        metadata={"applicant_id": applicant_id, "stage": stage, "outcome": outcome},
    )

print("log_stage() ready.")

log_stage() ready.


## 8. Define stage tools

In a real pipeline each stage calls external services (credit bureaus, doc verification APIs, income validators). For this notebook we stub them so the workflow runs end-to-end. The tools are decorated with `@tool` so the prebuilt agent can call them.

> **📌 The tools are deterministic stubs.** That's fine — the learning goal here is the workflow structure and the memory integration. In production, swap the stubs for real API calls and the rest of the notebook stays identical.

In [8]:
from langchain.tools import tool

@tool
def check_identity_documents(applicant_id: str) -> dict:
    """Verify identity documents for an applicant. Returns pass/fail + notes."""
    # Stub — production version would call a KYC provider
    stub = {
        "A-001": {"verified": True, "notes": "Driver's license + utility bill matched."},
        "A-002": {"verified": True, "notes": "Passport + bank statement matched."},
        "A-003": {"verified": False, "notes": "Utility bill address mismatch — requires re-submission."},
    }
    return stub.get(applicant_id, {"verified": False, "notes": "No documents on file."})

@tool
def pull_credit_report(applicant_id: str) -> dict:
    """Pull an applicant's credit score and summary. Returns score + bureau notes."""
    stub = {
        "A-001": {"score": 742, "notes": "No delinquencies in past 24 months. Experian."},
        "A-002": {"score": 598, "notes": "One 60-day delinquency in past 12 months. TransUnion."},
        "A-003": {"score": 710, "notes": "Thin file but no negatives. Equifax."},
    }
    return stub.get(applicant_id, {"score": 0, "notes": "No credit file."})

@tool
def verify_income_documents(applicant_id: str) -> dict:
    """Verify income from supporting documents. Returns monthly income + monthly debt."""
    stub = {
        "A-001": {"monthly_income": 11_500, "monthly_debt": 2_100, "notes": "2024 W-2 + 30 days pay stubs."},
        "A-002": {"monthly_income": 6_800, "monthly_debt": 3_400, "notes": "1040 return only; no pay stubs."},
        "A-003": {"monthly_income": 9_200, "monthly_debt": 1_600, "notes": "W-2 + pay stubs verified."},
    }
    return stub.get(applicant_id, {"monthly_income": 0, "monthly_debt": 0, "notes": "No income docs."})

print("Stage tools defined: check_identity_documents, pull_credit_report, verify_income_documents")

Stage tools defined: check_identity_documents, pull_credit_report, verify_income_documents


## 9. Build the workflow nodes

Each node is a plain Python function that takes the current state, does its work (sometimes by calling a prebuilt agent), writes an audit memory, and returns a partial state update. LangGraph merges the return value into the state automatically.

> **💡 Key insight:** For deterministic tasks (DTI calculation, threshold comparison) we use plain code — no LLM needed. For tasks that benefit from LLM reasoning over unstructured inputs (interpreting document verification results, writing a rationale), we use a `create_agent` instance. Mixing both inside a StateGraph is the point of this mode.

In [9]:
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="gpt-5.4", temperature=0.1)

# A small agent responsible for identity verification — it calls the tool and interprets the result
identity_agent = create_agent(
    model=model,
    tools=[check_identity_documents],
    system_prompt=(
        "You verify applicant identity. Call check_identity_documents with the applicant_id, "
        "then respond with exactly two lines: 'verified: true' or 'verified: false', and a one-sentence note."
    ),
)

credit_agent = create_agent(
    model=model,
    tools=[pull_credit_report],
    system_prompt=(
        "You pull and summarise credit. Call pull_credit_report for the applicant_id, then respond "
        "with two lines: 'score: <int>' and a one-sentence note on notable items."
    ),
)

income_agent = create_agent(
    model=model,
    tools=[verify_income_documents],
    system_prompt=(
        "You verify income. Call verify_income_documents, then respond with three lines: "
        "'monthly_income: <float>', 'monthly_debt: <float>', and a one-sentence note."
    ),
)

import re

def _extract(text: str, pattern: str, cast=str, default=None):
    m = re.search(pattern, text, re.IGNORECASE)
    if not m:
        return default
    try:
        return cast(m.group(1))
    except Exception:
        return default

def identity_node(state: MortgageState) -> dict:
    out = identity_agent.invoke({
        "messages": [{"role": "user",
                      "content": f"Verify applicant {state['applicant_id']}."}],
    })
    text = out["messages"][-1].content
    verified = "true" in text.lower().split("verified:", 1)[-1][:20]
    log_stage(state["applicant_id"], "identity",
              "pass" if verified else "fail", text)
    return {"identity_verified": verified, "identity_notes": text}

def credit_node(state: MortgageState) -> dict:
    out = credit_agent.invoke({
        "messages": [{"role": "user",
                      "content": f"Pull credit for applicant {state['applicant_id']}."}],
    })
    text = out["messages"][-1].content
    score = _extract(text, r"score:\s*(\d+)", cast=int, default=0)
    log_stage(state["applicant_id"], "credit", f"score={score}", text)
    return {"credit_score": score, "credit_notes": text}

def income_node(state: MortgageState) -> dict:
    out = income_agent.invoke({
        "messages": [{"role": "user",
                      "content": f"Verify income for applicant {state['applicant_id']}."}],
    })
    text = out["messages"][-1].content
    inc = _extract(text, r"monthly_income:\s*([\d.]+)", cast=float, default=0.0)
    dbt = _extract(text, r"monthly_debt:\s*([\d.]+)", cast=float, default=0.0)
    log_stage(state["applicant_id"], "income", f"inc={inc},debt={dbt}", text)
    return {"monthly_income": inc, "monthly_debt": dbt, "income_notes": text}

def dti_node(state: MortgageState) -> dict:
    inc = state.get("monthly_income", 0.0) or 0.0
    dbt = state.get("monthly_debt", 0.0) or 0.0
    dti = (dbt / inc) if inc > 0 else 1.0
    log_stage(state["applicant_id"], "dti", f"ratio={dti:.3f}",
              f"Computed DTI from monthly_income={inc:.2f}, monthly_debt={dbt:.2f}.")
    return {"dti_ratio": dti}

def decide_node(state: MortgageState) -> dict:
    score = state.get("credit_score", 0) or 0
    dti = state.get("dti_ratio", 1.0) or 1.0
    ident = state.get("identity_verified", False)
    loan = state.get("loan_amount", 0.0) or 0.0

    if not ident:
        decision, reason = "deny", "Identity not verified."
    elif score < 620:
        decision, reason = "deny", f"Credit score {score} below 620 minimum."
    elif dti > 0.43:
        decision, reason = "deny", f"DTI {dti:.2%} exceeds 43% ceiling."
    elif loan > 1_500_000:
        decision, reason = "manual_review", f"Loan ${loan:,.0f} above $1.5M jumbo threshold."
    else:
        decision, reason = "approve", f"All gates passed (score {score}, DTI {dti:.2%})."
    log_stage(state["applicant_id"], "decision", decision, reason)
    return {"decision": decision, "decision_reason": reason}

print("Nodes defined: identity_node, credit_node, income_node, dti_node, decide_node")

Nodes defined: identity_node, credit_node, income_node, dti_node, decide_node


## 10. Assemble the workflow as a `StateGraph`

Nodes become vertices; `add_edge` draws the mandatory arrows; `add_conditional_edges` draws the routed ones. A conditional edge is how we short-circuit the pipeline when identity verification fails — no point pulling credit for an applicant whose identity is unverified.

> **💡 Key insight:** Conditional edges are what turn a linear script into a real workflow. They let you encode the policy decisions ("if identity fails, skip to denial") in the graph topology instead of spreading if-statements across node bodies.

In [10]:
from langgraph.graph import StateGraph, START, END

def route_after_identity(state: MortgageState) -> str:
    return "credit" if state.get("identity_verified") else "decide"

def route_after_credit(state: MortgageState) -> str:
    return "income" if (state.get("credit_score") or 0) >= 620 else "decide"

builder = StateGraph(MortgageState)
builder.add_node("identity", identity_node)
builder.add_node("credit", credit_node)
builder.add_node("income", income_node)
builder.add_node("dti", dti_node)
builder.add_node("decide", decide_node)

builder.add_edge(START, "identity")
builder.add_conditional_edges("identity", route_after_identity,
                              {"credit": "credit", "decide": "decide"})
builder.add_conditional_edges("credit", route_after_credit,
                              {"income": "income", "decide": "decide"})
builder.add_edge("income", "dti")
builder.add_edge("dti", "decide")
builder.add_edge("decide", END)

graph = builder.compile()
print("Workflow compiled.")

Workflow compiled.


## 11. Run the workflow end-to-end

We run three applicants through the pipeline. Each produces a full audit trail in Oracle AI Agent Memory and ends with an explicit decision.

In [11]:
applicants = [
    {"applicant_id": "A-001", "applicant_name": "J. Patel",    "loan_amount":   480_000, "property_value":   600_000},
    {"applicant_id": "A-002", "applicant_name": "M. Johansson", "loan_amount":   320_000, "property_value":   400_000},
    {"applicant_id": "A-003", "applicant_name": "R. Okafor",   "loan_amount": 1_800_000, "property_value": 2_100_000},
]

for a in applicants:
    print(f"\n{'=' * 70}\nRunning workflow for {a['applicant_id']} ({a['applicant_name']})\n{'=' * 70}")
    final_state = graph.invoke(a)
    print(f"Decision: {final_state['decision'].upper()}")
    print(f"Reason:   {final_state['decision_reason']}")
    if final_state.get("credit_score"):
        print(f"  credit_score: {final_state['credit_score']}")
    if final_state.get("dti_ratio"):
        print(f"  dti: {final_state['dti_ratio']:.2%}")


Running workflow for A-001 (J. Patel)
Decision: APPROVE
Reason:   All gates passed (score 742, DTI 18.26%).
  credit_score: 742
  dti: 18.26%

Running workflow for A-002 (M. Johansson)
Decision: DENY
Reason:   Credit score 598 below 620 minimum.
  credit_score: 598

Running workflow for A-003 (R. Okafor)
Decision: DENY
Reason:   Identity not verified.


## 12. Stream a workflow run for debugging

`graph.stream(..., stream_mode="updates")` yields a dict per node as it executes. Great for notebooks and for surfacing stage-by-stage progress in an operator UI.

In [12]:
print("Streaming workflow for a new applicant:")
new_app = {
    "applicant_id": "A-004", "applicant_name": "D. Nguyen",
    "loan_amount": 525_000, "property_value": 650_000,
}
# Seed the fixtures for A-004 so stubs return data
# (in production this would be an external KYC/credit/income provider)
from langchain.tools import BaseTool  # just to show we're using langchain
# We'll simulate A-004 by monkey-patching the stubs for demonstration
_orig_identity = check_identity_documents.func
_orig_credit = pull_credit_report.func
_orig_income = verify_income_documents.func

def _identity_ext(applicant_id):
    if applicant_id == "A-004":
        return {"verified": True, "notes": "Passport + utility bill match."}
    return _orig_identity(applicant_id)

def _credit_ext(applicant_id):
    if applicant_id == "A-004":
        return {"score": 705, "notes": "Clean file, thin but positive."}
    return _orig_credit(applicant_id)

def _income_ext(applicant_id):
    if applicant_id == "A-004":
        return {"monthly_income": 10_200, "monthly_debt": 2_900, "notes": "W-2 + pay stubs verified."}
    return _orig_income(applicant_id)

check_identity_documents.func = _identity_ext
pull_credit_report.func = _credit_ext
verify_income_documents.func = _income_ext

for chunk in graph.stream(new_app, stream_mode="updates"):
    for node_name, update in chunk.items():
        print(f"  [{node_name}] -> {list(update.keys())}")

Streaming workflow for a new applicant:
  [identity] -> ['identity_verified', 'identity_notes']
  [credit] -> ['credit_score', 'credit_notes']
  [income] -> ['monthly_income', 'monthly_debt', 'income_notes']
  [dti] -> ['dti_ratio']
  [decide] -> ['decision', 'decision_reason']


## 13. Inspect the audit trail for one applicant

Every stage wrote a structured memory with `applicant_id` metadata. Pulling the trail is one memory search away.

In [13]:
# Get the complete stage trail for A-001 via metadata filter
trail = store.list(
    "memory",
    user_id=UNDERWRITER_ID,
    agent_id=AGENT_ID,
    metadata_filter={"applicant_id": "A-001"},
    limit=50,
)
print(f"Audit trail for A-001 ({len(trail)} entries):")
for r in trail:
    stage = (r.metadata or {}).get("stage", "?")
    outcome = (r.metadata or {}).get("outcome", "?")
    print(f"  [{stage:<8}] [{outcome}] {r.content[:100]}")

Audit trail for A-001 (10 entries):
  [identity] [pass] [identity] applicant=A-001 outcome=pass. verified: true  
The identity documents for applicant A-001
  [credit  ] [score=742] [credit] applicant=A-001 outcome=score=742. score: 742  
Notable items: No delinquencies in the past
  [income  ] [inc=11500.0,debt=2100.0] [income] applicant=A-001 outcome=inc=11500.0,debt=2100.0. monthly_income: 11500.0  
monthly_debt: 21
  [dti     ] [ratio=0.183] [dti] applicant=A-001 outcome=ratio=0.183. Computed DTI from monthly_income=11500.00, monthly_debt=2
  [decision] [approve] [decision] applicant=A-001 outcome=approve. All gates passed (score 742, DTI 18.26%).
  [identity] [pass] [identity] applicant=A-001 outcome=pass. verified: true
Driver's license and utility bill matched.
  [credit  ] [score=742] [credit] applicant=A-001 outcome=score=742. score: 742
note: No delinquencies in the past 24 months;
  [income  ] [inc=11500.0,debt=2100.0] [income] applicant=A-001 outcome=inc=11500.0,debt=2100.0

## 14. Recall a policy fact

Because we stored underwriting thresholds as `fact` records, any node — or any downstream reporting tool — can recall them with semantic search. Policy changes become a memory update rather than a code change.

In [14]:
policy_hits = memory_client.search(
    "maximum debt to income ratio",
    user_id=UNDERWRITER_ID, agent_id=AGENT_ID,
    record_types=["fact"], max_results=3,
)
print("Relevant policy facts:")
for r in policy_hits:
    print(f"  - {r.content}  [distance={r.distance:.3f}]")

/Users/richmondalake/Desktop/projects/OAMP/.venv313/lib/python3.13/site-packages/oracleagentmemory/core/oracleagentmemory.py:698: UserWarning: You are calling an asynchronous method in a synchronous method from an asynchronous context. This is highly discouraged because it can lead to deadlocks. Please use the asynchronous method equivalent: 
  return run_async_in_sync(self._do_search_async, query, scope, max_results, record_types)


Relevant policy facts:
  - Maximum acceptable debt-to-income ratio is 43%.  [distance=0.226]
  - Maximum acceptable debt-to-income ratio is 43%.  [distance=0.226]
  - Minimum acceptable credit score for conventional loans is 620.  [distance=0.652]


## 15. Cleanup

In [15]:
connection.close()
print("Connection closed.")

Connection closed.


## Key Takeaways

> **🎯 1. Workflow mode is deterministic orchestration over non-deterministic reasoning.** The graph topology is fixed; the LLM only does local reasoning inside nodes. That split is what makes workflows auditable in regulated domains.

> **🎯 2. `create_agent` inside a `StateGraph` gives you the best of both worlds.** You get a deterministic pipeline you can debug and a prebuilt agent that handles tool-calling and message management inside each stage. Don't reinvent the agent loop; use it as a node.

> **🎯 3. Memory is your audit trail.** Every stage's decision and rationale should be persisted with structured metadata (`applicant_id`, `stage`, `outcome`). That turns semantic search into a file-review tool and turns metadata filters into a per-applicant audit viewer.

> **🎯 4. Policy belongs in memory, not in code.** Underwriting thresholds, accepted document types, compliance rules — store them as `fact` records. Policy teams can update them without a code change, and every agent node can recall them via search.

> **🎯 5. Oracle AI Database is the right substrate for regulated workflows.** The audit trail, the applicant state, the policy facts, and the conversation transcripts all live in one governed backend — the same one the rest of the business already uses. That collapses four compliance reviews into one.